# Ariane - LLM-guided **LEGAL** placement of all 932 macros (integrated greedy)\n\nAll 932 macros are placed by the **same wiremask greedy** with the position mask, so the layout is **legal by construction** (zero overlap, in-canvas) -- no soft-fill, no separate legalizer. At grid **448** all 932 fit (only ~814/932 at 224). The LLM reorders the top-N most-connected **hubs**; the rest keep topology order. We compare **heuristic baseline vs random hub control vs LLM hub search**, all legal, against MaskPlace's paper number.\n\nThis notebook is independent of `ariane932twostage.ipynb` and uses only the additive `integrated_search.py` module.

## Cell 1 - Setup (clone repo, checkout integrated branch, gym/protobuf/anthropic)

In [ ]:
import subprocess, sys, os, shutil, types, glob
import numpy as np

REPO_URL  = "https://github.com/dennis5727/arianePlacement.git"
BRANCH    = "integrated-932-search"   # branch carrying integrated_search.py (merge to main when ready)
CLONE_DIR = "/kaggle/working/arianePlacement"
WORK_DIR  = os.path.join(CLONE_DIR, "maskplace")

if not os.path.exists(CLONE_DIR):
    try:
        subprocess.check_call(["git","clone",REPO_URL,CLONE_DIR]); print("cloned")
    except Exception as e: print("git clone failed:", e)
else:
    subprocess.call(["git","-C",CLONE_DIR,"fetch","--all"]); print("fetched")
# use the integrated branch if present; otherwise fall back to the default branch
if subprocess.call(["git","-C",CLONE_DIR,"checkout",BRANCH]) == 0:
    subprocess.call(["git","-C",CLONE_DIR,"pull","--ff-only"]); print("on branch", BRANCH)
else:
    print("branch", BRANCH, "not found -- push it or merge to main")

if not os.path.exists(WORK_DIR):
    hits=glob.glob("/kaggle/input/**/place_db.py",recursive=True)
    assert hits,"no code via git or dataset"
    shutil.copytree(os.path.dirname(hits[0]),WORK_DIR); print("dataset fallback")

os.chdir(WORK_DIR); sys.path.insert(0,WORK_DIR); print("cwd:",os.getcwd())

os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION","python")
try: subprocess.check_call([sys.executable,"-m","pip","install","-q","protobuf==3.20.3"])
except subprocess.CalledProcessError as e: print("protobuf pin failed:",e)

try:
    import gym; print("real gym:",gym.__version__)
except Exception:
    gym=types.ModuleType("gym"); spaces=types.ModuleType("gym.spaces")
    reg=types.ModuleType("gym.envs.registration"); envs=types.ModuleType("gym.envs")
    class _E: pass
    class _D:
        def __init__(s,n): s.n=int(n)
        def contains(s,x):
            try: x=int(x)
            except: return False
            return 0<=x<s.n
    class _B:
        def __init__(s,low=0,high=1,shape=None,dtype=None): s.low,s.high,s.shape,s.dtype=low,high,shape,dtype
    gym.Env=_E; spaces.Discrete=_D; spaces.Box=_B; gym.spaces=spaces
    reg.register=lambda *a,**k:None; gym.envs=envs
    sys.modules.update({"gym":gym,"gym.spaces":spaces,"gym.envs":envs,"gym.envs.registration":reg})
    print("gym shim")

try: subprocess.check_call([sys.executable,"-m","pip","install","-q","anthropic"])
except subprocess.CalledProcessError as e: print("anthropic skipped:",e)

need=["place_db.py","place_env/place_env.py","comp_res.py","greedy_place.py",
      "parse_netlist.py","region_constraint.py","strong_search.py","trade_off_eval.py",
      "integrated_search.py","ariane/netlist.pb.txt"]
miss=[f for f in need if not os.path.exists(f)]
assert not miss, f"MISSING (push branch {BRANCH}): {miss}"
print("files OK\n=== SETUP COMPLETE ===")

## Cell 2 - Sanity (932 = 133 hard + 799 soft) + load PlaceDB

In [ ]:
from place_db import PlaceDB
placedb = PlaceDB("ariane")
hard=[n for n in placedb.node_info if placedb.node_info[n].get("is_hard")]
soft=[n for n in placedb.node_info if not placedb.node_info[n].get("is_hard")]
print("Nodes", len(placedb.node_info), "| hard", len(hard), "| soft", len(soft),
      "| Nets", len(placedb.net_info), "| Ports", len(placedb.port_info))
assert len(placedb.node_info) == 932
print("sanity OK")

## Cell 3 - API key (needed only for the LLM cell)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["ANTHROPIC_API_KEY"] = UserSecretsClient().get_secret("ANTHROPIC_API_KEY")
print("API key loaded")

## Cell 4 - Config

In [ ]:
import importlib, integrated_search
importlib.reload(integrated_search)
import integrated_search as isr

TOP_N      = 932                # ALL 932 macros are reorder-able (the LLM/random may target ANY).
                                # set to e.g. 300 to restrict to the most-connected hubs (smaller prompt).
GRID       = 448               # integrated greedy fits all 932 LEGALLY at 448 (only 814/932 at 224)
MODEL      = "claude-opus-4-8"  # strongest model; ~$0.1-0.2 per run (tiny output, cached prompt)
MAX_ITERS  = 8
PATIENCE   = 3
N_RANDOM   = 12
LLM_ACTION = "promote"          # option C: targeted 'place A before B' moves on the topology order
                                # ("hub" = move hubs to front, weaker -- usually does NOT beat topology).
print(f"action={LLM_ACTION} | reorder-able macros: top {TOP_N} of 932 | grid {GRID} | model {MODEL}")
print("NOTE: each integrated 932-greedy at 448 takes ~60-150s; with TOP_N=932 the LLM prompt is")
print("      larger (all 932 macros listed, cached). Searches run for MANY minutes.")

## Cell 5 - Heuristic baseline + random hub control (FREE, no API) -- LEGAL all 932

In [ ]:
# FREE (no API): heuristic baseline (topology order) + matched no-LLM random control.
# Both place ALL 932 with the position-mask greedy -> LEGAL (zero overlap) by construction.
base = isr.heuristic_baseline(placedb, grid=GRID, verbose=True)
if LLM_ACTION == "promote":
    rc = isr.random_promote_control(placedb, top_n=TOP_N, grid=GRID, n_evals=N_RANDOM, verbose=True)
else:
    rc = isr.random_hub_control(placedb, top_n=TOP_N, grid=GRID, n_evals=N_RANDOM, verbose=True)

## Cell 6 - LLM search (PAID, text) -- targeted edits to topology (option C) -- LEGAL all 932


In [ ]:
# PAID (text): the LLM improves the topology order; every candidate scored on the integrated
# LEGAL full-932 layout (accept-if-improves, floored at the topology baseline -> never worse).
if LLM_ACTION == "promote":
    res = isr.llm_promote_search(placedb, top_n=TOP_N, grid=GRID, model=MODEL,
                                 max_iters=MAX_ITERS, patience=PATIENCE, verbose=True)
else:
    res = isr.llm_hub_search(placedb, top_n=TOP_N, grid=GRID, model=MODEL,
                             max_iters=MAX_ITERS, patience=PATIENCE, verbose=True)

## Cell 7 - Results table (legality + MaskPlace comparison) + CSV

In [ ]:
# Combined legality + comparison table (run after whichever cells above you ran).
# Reports BOTH bbox HPWL and MST; compare the MST column to MaskPlace (its number is MST).
import csv
from trade_off_eval import usd_cost

def _row(method, r, calls=0, intok=0, outok=0, ctok=0):
    ov, oob = isr.overlaps_and_oob(r["best_env"], GRID)
    return dict(method=method, hpwl=r["hpwl"], mst=isr.mst_of(placedb, r["best_env"]),
                overlaps=ov, out_of_canvas=oob, macros=len(r["best_env"].node_pos), grid=GRID,
                calls=calls, out_tokens=outok,
                usd=(usd_cost(MODEL, intok, outok, ctok) if calls else 0.0))

ctrl_label = "random promote ctrl" if LLM_ACTION == "promote" else "random hub control"
rows = []
if "base" in globals(): rows.append(_row("heuristic baseline", base))
if "rc"   in globals(): rows.append(_row(ctrl_label, rc))
if "res"  in globals(): rows.append(_row(f"LLM {LLM_ACTION}", res, calls=res["calls"],
                                         intok=res["in_tokens"], outok=res["out_tokens"],
                                         ctok=res["cache_read_tokens"]))
rows.append(dict(method="MaskPlace RL (paper)", hpwl=None, mst=isr.MASKPLACE_ARIANE_MST,
                 overlaps=0, out_of_canvas=0, macros=932, grid=224, calls=0, out_tokens=0, usd=None))

def _f(v): return f"{v:.3e}" if v is not None else "-"
print(f"\n{'method':<22}{'HPWL(bbox)':>12}{'MST':>12}{'macros':>8}{'overlaps':>9}{'oob':>5}{'grid':>6}{'$':>8}{'out_tok':>9}")
print("-"*92)
for r in rows:
    usd = f"{r['usd']:.3f}" if r['usd'] is not None else "-"
    inc = r['macros'] < 932 and r['method'] != "MaskPlace RL (paper)"   # incomplete -> HPWL not comparable
    hb = "incompl" if inc else _f(r['hpwl'])
    mb = "incompl" if inc else _f(r['mst'])
    print(f"{r['method']:<22}{hb:>12}{mb:>12}{r['macros']:>8}{r['overlaps']:>9}"
          f"{r['out_of_canvas']:>5}{r['grid']:>6}{usd:>8}{r['out_tokens']:>9}")
print("\nMST column is metric-matched to MaskPlace (its 1.463e6 is MST, grid 224). Our rows are")
print("legal at grid 448. 'incompl'=<932 macros placed (HPWL not comparable). Judge the LLM vs the control.")

with open("/kaggle/working/integrated_932.csv","w",newline="") as f:
    w=csv.DictWriter(f, fieldnames=["method","hpwl","mst","overlaps","out_of_canvas","macros","grid","calls","out_tokens","usd"])
    w.writeheader(); w.writerows(rows)
print("saved /kaggle/working/integrated_932.csv")